**1. Setup & Installation**

In [1]:
# Install Chef's Hat Gym from PyPI
import subprocess, sys

packages = [
    'ChefsHatGym==3.0.0.1',
    'gym==0.26.2',      # ChefsHatGym requires gym >=0.26 (5-tuple step API)
    'numpy>=1.24,<2.0', # pin numpy to avoid binary incompatibility with gym/pandas
    'tensorflow',
    'matplotlib',
    'pandas',
]

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

print('Installation complete.')


Installation complete.


In [2]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

# Compatibility shim: gym uses np.bool8 which was removed in NumPy 2.0
if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Dense, Lambda, Add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber
from tensorflow.keras.models import load_model as keras_load_model
from collections import deque

# Chef's Hat Gym – correct package paths (PyPI ChefsHatGym==3.0.0.1)
from ChefsHatGym.gameRooms.chefs_hat_room_local import ChefsHatRoomLocal
from ChefsHatGym.agents.base_classes.chefs_hat_player import ChefsHatPlayer
from ChefsHatGym.agents.agent_random import AgentRandon
from ChefsHatGym.env.ChefsHatEnv import ChefsHatEnv, GAMETYPE
from ChefsHatGym.rewards.only_winning import RewardOnlyWinning

# Reproducibility helper
def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

STUDENT_ID = 16747960
VARIANT    = STUDENT_ID % 7
BASE_SEED  = 42
set_global_seed(BASE_SEED)

print(f'Student ID: {STUDENT_ID}  |  Variant: {VARIANT}  (Robustness & Generalisation)')
print(f'TensorFlow: {tf.__version__}')


/tmp/ipykernel_7317/1626837575.py:10: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not hasattr(np, 'bool8'):
I0000 00:00:1780241847.621741    7317 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780241849.695263    7317 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Student ID: 16747960  |  Variant: 5  (Robustness & Generalisation)
TensorFlow: 2.21.0


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


**2. DQN Agent**

In [3]:
def _dueling_mean(a):
    return a - tf.reduce_mean(a, axis=1, keepdims=True)


def build_dueling_dqn(state_size: int, action_size: int, lr: float) -> Model:
    """Dueling DQN: shared layers → value stream + advantage stream."""
    inp   = Input(shape=(state_size,))
    x     = Dense(256, activation='relu')(inp)
    x     = Dense(128, activation='relu')(x)
    x     = Dense(64,  activation='relu')(x)
    value = Dense(1,           activation='linear')(x)
    adv   = Dense(action_size, activation='linear')(x)
    adv   = Lambda(_dueling_mean, output_shape=(action_size,))(adv)
    q_out = Add()([value, adv])
    model = Model(inp, q_out)
    model.compile(loss=Huber(), optimizer=Adam(learning_rate=lr))
    return model


class DQNAgent(ChefsHatPlayer):
    """
    Dueling Double DQN agent for ChefsHatGym V3 (PyPI).

    Observation layout (228 elements):
        [0:11]   board cards
        [11:28]  hand cards
        [28:228] one-hot action mask (valid actions == 1)

    State fed to network: [board/13, hand/13]  (28 features)
    Action returned: one-hot numpy array of length 200
    """

    suffix     = "DQN"
    STATE_SIZE  = 28
    ACTION_SIZE = 200

    def __init__(
        self,
        name: str = 'DQN',
        *,
        gamma: float         = 0.99,
        lr: float            = 1e-4,
        epsilon: float       = 1.0,
        epsilon_min: float   = 0.05,
        epsilon_decay: float = 0.995,
        batch_size: int      = 256,
        memory_size: int     = 10_000,
        training: bool       = True,
        model_path: str      = None,
        load_model: bool     = False,
        log_directory: str   = '',
        verbose_console: bool = False,
    ):
        super().__init__(
            self.suffix, name,
            log_directory, verbose_console, False,
            log_directory, True,
        )
        self.gamma         = gamma
        self.epsilon       = epsilon if training else 0.0
        self.epsilon_min   = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.batch_size    = batch_size
        self.training      = training
        self.model_path    = model_path

        self.memory   = deque(maxlen=memory_size)
        self.reward_f = RewardOnlyWinning()

        self._episode:        list = []
        self._last_state            = None
        self._last_action_idx       = None
        self._last_mask             = None

        self._my_player_idx   = -1
        self._prev_game_score = 0

        self.loss_history:      list = []
        self.epsilon_history:   list = []
        self.positions:         list = []
        self.rewards_per_match: list = []

        if load_model and model_path and os.path.exists(model_path):
            co = {'_dueling_mean': _dueling_mean}
            self.model        = keras_load_model(model_path, custom_objects=co)
            target_p          = model_path.replace('.h5', '.target.h5')
            self.target_model = (keras_load_model(target_p, custom_objects=co)
                                 if os.path.exists(target_p)
                                 else build_dueling_dqn(self.STATE_SIZE, self.ACTION_SIZE, lr))
        else:
            self.model        = build_dueling_dqn(self.STATE_SIZE, self.ACTION_SIZE, lr)
            self.target_model = build_dueling_dqn(self.STATE_SIZE, self.ACTION_SIZE, lr)
            self.target_model.set_weights(self.model.get_weights())

    def get_action(self, observations):
        obs       = np.array(observations, dtype=np.float32)
        board     = obs[0:11]  / 13.0
        hand      = obs[11:28] / 13.0
        state     = np.concatenate([board, hand])
        mask      = obs[28:]

        action_idx = self._select_action(state, mask)

        shaped_r = -0.02
        if mask[action_idx] == 0:
            shaped_r -= 1.0

        if self._last_state is not None and self.training:
            self._episode.append((
                self._last_state, self._last_mask,
                self._last_action_idx, shaped_r,
                state, mask, False,
            ))

        self._last_state      = state
        self._last_action_idx = action_idx
        self._last_mask       = mask

        a = np.zeros(self.ACTION_SIZE, dtype=int)
        a[action_idx] = 1
        return a

    def get_exhanged_cards(self, cards, amount):
        return sorted(cards)[-amount:]

    def do_special_action(self, info, specialAction):
        return True

    def update_my_action(self, envInfo):
        pass

    def update_action_others(self, envInfo):
        pass

    def update_start_match(self, cards, players, starting_player):
        my_name = self.get_name()
        if my_name in players:
            self._my_player_idx = players.index(my_name)
        self._episode.clear()
        self._last_state      = None
        self._last_action_idx = None
        self._last_mask       = None

    def update_exchange_cards(self, cards_sent, cards_received):
        pass

    def update_end_match(self, envInfo):
        reward, place = self._compute_final_reward(envInfo)
        self.rewards_per_match.append(reward)
        self.positions.append(place)

        if self._last_state is not None and self.training:
            self._episode.append((
                self._last_state, self._last_mask,
                self._last_action_idx, reward,
                self._last_state, self._last_mask, True,
            ))

        for exp in self._episode:
            self.memory.append(exp)
        self._episode.clear()
        self._replay()

    def get_reward(self, envInfo):
        try:
            game_scores = envInfo.get("Game_Score", [])
            idx = self._my_player_idx
            if idx < 0 or idx >= len(game_scores):
                return -0.001
            delta = int(game_scores[idx]) - self._prev_game_score
            is_over = all(envInfo.get("Finished_Players", [False]))
            return self.reward_f.getReward(
                thisPlayerPosition=delta,
                matchFinished=is_over,
            )
        except Exception:
            return -0.001

    def _compute_final_reward(self, envInfo):
        try:
            game_scores = envInfo.get("Game_Score", [])
            idx = self._my_player_idx
            if idx < 0 or idx >= len(game_scores):
                return -1.0, 4
            current_game_score = int(game_scores[idx])
            delta = current_game_score - self._prev_game_score
            self._prev_game_score = current_game_score
            delta_to_place  = {3: 1, 2: 2, 1: 3, 0: 4}
            delta_to_reward = {3: 1.0, 2: 0.3, 1: -0.3, 0: -1.0}
            place  = delta_to_place.get(delta, 4)
            reward = delta_to_reward.get(delta, -1.0)
        except Exception:
            place, reward = 4, -1.0
        return reward, place

    def _select_action(self, state, mask):
        valid_idxs = np.where(mask == 1)[0].tolist()
        if not valid_idxs:
            return 0

        if self.training and np.random.rand() < self.epsilon:
            return int(np.random.choice(valid_idxs))

        q        = self.model.predict(state[np.newaxis, :], verbose=0)[0]
        masked_q = np.where(mask, q, -np.inf)
        if np.all(masked_q == -np.inf):
            return int(np.random.choice(valid_idxs))
        return int(np.argmax(masked_q))

    def _soft_update(self, tau: float = 0.05):
        new_w = [tau * m + (1 - tau) * t
                 for m, t in zip(self.model.get_weights(),
                                 self.target_model.get_weights())]
        self.target_model.set_weights(new_w)

    def _replay(self):
        if not self.training or len(self.memory) < self.batch_size:
            return

        batch = random.sample(self.memory, self.batch_size)
        S, M, A, R, S2, M2, D = map(np.array, zip(*batch))

        q_next       = self.model.predict(S2, verbose=0)
        q_next_mask  = np.where(M2, q_next, -np.inf)
        best_next    = np.argmax(q_next_mask, axis=1)
        q_target_net = self.target_model.predict(S2, verbose=0)

        target = self.model.predict(S, verbose=0)
        for i in range(self.batch_size):
            target[i][A[i]] = (R[i] if D[i]
                               else R[i] + self.gamma * q_target_net[i][best_next[i]])

        h = self.model.fit(S, target, epochs=1, verbose=0)
        self.loss_history.append(float(np.mean(h.history['loss'])))
        self.epsilon_history.append(self.epsilon)

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
        self._soft_update()

    def save_model(self):
        if self.model_path:
            os.makedirs(os.path.dirname(self.model_path), exist_ok=True)
            self.model.save(self.model_path)


print('DQNAgent class defined.')


DQNAgent class defined.


**3. Greedy Agent**

In [4]:
class GreedyAgent(ChefsHatPlayer):
    """
    Greedy heuristic agent: plays the valid action with the highest card sum.
    Observation layout same as DQNAgent.
    """
    suffix = "GREEDY"

    def __init__(self, name: str, log_directory: str = '', verbose_console: bool = False):
        super().__init__(
            self.suffix, name,
            log_directory, verbose_console, False,
            log_directory, True,
        )
        self.reward_f = RewardOnlyWinning()

    def get_action(self, observations):
        obs          = np.array(observations, dtype=np.float32)
        action_mask  = obs[28:]
        valid_idxs   = np.where(action_mask == 1)[0].tolist()
        if not valid_idxs:
            chosen = 0
        else:
            # Prefer action with highest index (heuristic: higher index ≈ higher card value)
            chosen = max(valid_idxs)
        a = np.zeros(200, dtype=int)
        a[chosen] = 1
        return a

    def get_exhanged_cards(self, cards, amount):
        return sorted(cards)[-amount:]

    def do_special_action(self, info, specialAction):
        return True

    def update_my_action(self, envInfo):
        pass

    def update_action_others(self, envInfo):
        pass

    def update_start_match(self, cards, players, starting_player):
        pass

    def update_exchange_cards(self, cards_sent, cards_received):
        pass

    def update_end_match(self, envInfo):
        pass

    def get_reward(self, envInfo):
        return self.reward_f.getReward(envInfo)


print('GreedyAgent class defined.')


GreedyAgent class defined.


**4. Training Infrastructure**

In [5]:
def run_game(
    agent: DQNAgent,
    opponents: list,
    n_matches: int,
    output_folder: str,
    room_name: str = 'ChefsHatRoom',
) -> dict:
    """Run one game (n_matches) with ChefsHatRoomLocal and return metrics."""
    os.makedirs(output_folder, exist_ok=True)

    room = ChefsHatRoomLocal(
        room_name=room_name,
        game_type=GAMETYPE['MATCHES'],
        stop_criteria=n_matches,
        save_dataset=True,
        verbose_console=False,
        verbose_log=False,
        game_verbose_console=False,
        game_verbose_log=False,
        log_directory=output_folder,
    )

    for opp in opponents:
        room.add_player(opp)
    room.add_player(agent)

    room.start_new_game()

    positions = agent.positions.copy()
    rewards   = agent.rewards_per_match.copy()

    return {
        'positions':       positions,
        'rewards':         rewards,
        'loss_history':    agent.loss_history.copy(),
        'epsilon_history': agent.epsilon_history.copy(),
        'win_rate':        positions.count(1) / max(len(positions), 1),
        'mean_position':   float(np.mean(positions)) if positions else 4.0,
        'mean_reward':     float(np.mean(rewards))   if rewards   else 0.0,
    }


def make_random_opponents(n: int, log_dir: str) -> list:
    os.makedirs(log_dir, exist_ok=True)
    return [AgentRandon(name=f'Rnd{i}', log_directory=log_dir,
                        verbose_console=False, verbose_log=False)
            for i in range(n)]


def make_greedy_opponents(n: int, log_dir: str) -> list:
    os.makedirs(log_dir, exist_ok=True)
    return [GreedyAgent(name=f'Grdy{i}', log_directory=log_dir,
                        verbose_console=False)
            for i in range(n)]


print('Training infrastructure ready.')


Training infrastructure ready.


**5. Baseline Training**

In [ ]:
BASELINE_MATCHES = 200
OUTPUT_BASE      = 'outputs/baseline'
MODEL_PATH       = os.path.join(OUTPUT_BASE, 'model', 'dqn_model.h5')
os.makedirs(OUTPUT_BASE, exist_ok=True)

set_global_seed(BASE_SEED)

baseline_agent = DQNAgent(
    name='DQN_Baseline',
    gamma=0.99,
    lr=1e-4,
    epsilon=1.0,
    epsilon_min=0.05,
    epsilon_decay=0.995,
    batch_size=256,
    memory_size=10_000,
    training=True,
    model_path=MODEL_PATH,
    log_directory=OUTPUT_BASE,
    verbose_console=False,
)

dummy_room_dir = os.path.join(OUTPUT_BASE, 'agents_tmp')
opponents_baseline = make_random_opponents(3, dummy_room_dir)

print(f'Training DQN for {BASELINE_MATCHES} matches vs 3 Random agents ...')
baseline_results = run_game(
    baseline_agent,
    opponents_baseline,
    n_matches=BASELINE_MATCHES,
    output_folder=OUTPUT_BASE,
    room_name='Baseline_Train',
)

print(f"Baseline training complete.")


In [7]:
def plot_learning_curve(results: dict, title: str, window: int = 20):
    positions = np.array(results['positions'])
    rewards   = np.array(results['rewards'])
    losses    = np.array(results['loss_history'])

    def rolling(x, w):
        if len(x) < w:
            return x, np.arange(len(x))
        r = np.convolve(x, np.ones(w) / w, mode='valid')
        return r, np.arange(w - 1, len(x))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # 1 – Finishing position
    ax = axes[0]
    ax.plot(positions, alpha=0.3, color='steelblue', label='Raw')
    r, x = rolling(positions, window)
    ax.plot(x, r, color='steelblue', lw=2, label=f'MA{window}')
    ax.invert_yaxis()
    ax.set(xlabel='Match', ylabel='Finishing position (1=best)', title='Finishing Position')
    ax.legend()

    # 2 – Reward
    ax = axes[1]
    ax.plot(rewards, alpha=0.3, color='darkorange', label='Raw')
    r, x = rolling(rewards, window)
    ax.plot(x, r, color='darkorange', lw=2, label=f'MA{window}')
    ax.set(xlabel='Match', ylabel='Reward', title='Reward per Match')
    ax.legend()

    # 3 – Training loss
    ax = axes[2]
    ax.plot(losses, color='crimson', lw=1, label='Loss')
    ax.set(xlabel='Replay step', ylabel='Huber loss', title='Training Loss')
    ax.legend()

    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


plot_learning_curve(baseline_results, 'Baseline DQN – Training vs 3 Random Opponents')

**6. Robustness Experiment 1 – Multiple Random Seeds**

In [ ]:
SEED_MATCHES = 100
SEEDS        = [0, 7, 21, 42, 99]
seed_results = []

for seed in SEEDS:
    set_global_seed(seed)
    folder = f'outputs/seed_{seed}'
    mpath  = os.path.join(folder, 'model', 'dqn_model.h5')
    os.makedirs(folder, exist_ok=True)

    agent = DQNAgent(
        name=f'DQN_s{seed}',
        training=True,
        model_path=mpath,
        log_directory=folder,
        verbose_console=False,
    )
    opps = make_random_opponents(3, os.path.join(folder, 'atmp'))

    print(f'  Seed {seed:3d} ... ', end='', flush=True)
    res = run_game(agent, opps, SEED_MATCHES, folder, room_name=f'Seed_{seed}')
    seed_results.append({'seed': seed, **res})
    print(f"win_rate={res['win_rate']:.3f}  mean_pos={res['mean_position']:.2f}")

print('\nSeed experiment complete.')


In [9]:
# Summary statistics across seeds
seed_df = pd.DataFrame([
    {'Seed': r['seed'],
     'Win Rate': r['win_rate'],
     'Mean Position': r['mean_position'],
     'Mean Reward': r['mean_reward']}
    for r in seed_results
])

print(seed_df.to_string(index=False))
print(f"\nWin Rate  – mean: {seed_df['Win Rate'].mean():.3f}  std: {seed_df['Win Rate'].std():.3f}")
print(f"Mean Pos  – mean: {seed_df['Mean Position'].mean():.2f}  std: {seed_df['Mean Position'].std():.3f}")

 Seed  Win Rate  Mean Position  Mean Reward
    0      0.39           2.01        0.320
    7      0.39           2.05        0.296
   21      0.28           2.41        0.059
   42      0.35           2.17        0.216
   99      0.33           2.24        0.171

Win Rate  – mean: 0.348  std: 0.046
Mean Pos  – mean: 2.18  std: 0.160


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

window = 20
colors = plt.cm.tab10(np.linspace(0, 0.5, len(SEEDS)))

for r, col in zip(seed_results, colors):
    pos  = np.array(r['positions'])
    if len(pos) >= window:
        smooth = np.convolve(pos, np.ones(window)/window, mode='valid')
        axes[0].plot(smooth, color=col, label=f"seed={r['seed']}")

axes[0].invert_yaxis()
axes[0].set(xlabel='Match', ylabel='Avg finishing position', title='Position per Seed (MA-20)')
axes[0].legend(fontsize=8)

# Bar chart: win rate per seed
axes[1].bar(
    [str(r['seed']) for r in seed_results],
    [r['win_rate'] for r in seed_results],
    color=colors,
)
axes[1].axhline(0.25, ls='--', color='grey', label='Random baseline (25%)')
axes[1].set(xlabel='Seed', ylabel='Win rate', title='Win Rate per Seed')
axes[1].legend()

plt.suptitle('Experiment 1 – Robustness Across Random Seeds', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**7. Robustness Experiment 2 – Generalisation Across Opponent Types**

In [ ]:
GENERALISATION_MATCHES = 100

def train_agent(name, opponent_factory, n_matches, out_dir):
    """Train a fresh DQN agent and return it."""
    mpath = os.path.join(out_dir, 'model', 'dqn_model.h5')
    agent = DQNAgent(name=name, training=True, model_path=mpath,
                     log_directory=out_dir, verbose_console=False)
    opps  = opponent_factory(3, os.path.join(out_dir, 'atmp'))
    run_game(agent, opps, n_matches, out_dir, room_name=f'{name}_train')
    return agent


def eval_agent(agent, opponent_factory, n_matches, out_dir, suffix=''):
    """Evaluate a trained agent (no further learning)."""
    agent.training = False
    agent.epsilon  = 0.0
    # reset metrics for eval run only
    agent.positions.clear()
    agent.rewards_per_match.clear()
    opps = opponent_factory(3, os.path.join(out_dir, 'atmp' + suffix))
    res  = run_game(agent, opps, n_matches, out_dir,
                    room_name=f'{agent.name}_eval{suffix}')
    agent.training = True
    return res


set_global_seed(BASE_SEED)

# Condition A: trained vs Random
print('Training agent_vs_random ...')
agent_rnd = train_agent('DQN_vs_Rnd', make_random_opponents,
                         GENERALISATION_MATCHES, 'outputs/gen_rnd_train')

# Condition C: trained vs Greedy
print('Training agent_vs_greedy ...')
agent_grdy = train_agent('DQN_vs_Grdy', make_greedy_opponents,
                          GENERALISATION_MATCHES, 'outputs/gen_grdy_train')

print('Training complete.')

In [12]:
set_global_seed(BASE_SEED)

EVAL_MATCHES = 50

# Condition A – in-distribution: trained vs Random → eval vs Random
res_AA = eval_agent(agent_rnd,  make_random_opponents, EVAL_MATCHES,
                     'outputs/gen_eval', suffix='_AA')

# Condition B – out-of-distribution: trained vs Random → eval vs Greedy
res_AB = eval_agent(agent_rnd,  make_greedy_opponents, EVAL_MATCHES,
                     'outputs/gen_eval', suffix='_AB')

# Condition C – in-distribution: trained vs Greedy → eval vs Greedy
res_CC = eval_agent(agent_grdy, make_greedy_opponents, EVAL_MATCHES,
                     'outputs/gen_eval', suffix='_CC')

# Summary table
gen_df = pd.DataFrame([
    {'Condition': 'A: Rnd→Rnd (in-dist)',   'Win Rate': res_AA['win_rate'],
     'Mean Pos': res_AA['mean_position'],    'Mean Reward': res_AA['mean_reward']},
    {'Condition': 'B: Rnd→Grdy (out-dist)', 'Win Rate': res_AB['win_rate'],
     'Mean Pos': res_AB['mean_position'],    'Mean Reward': res_AB['mean_reward']},
    {'Condition': 'C: Grdy→Grdy (in-dist)', 'Win Rate': res_CC['win_rate'],
     'Mean Pos': res_CC['mean_position'],    'Mean Reward': res_CC['mean_reward']},
])

print(gen_df.to_string(index=False))


Agent RANDOM_Rnd0 log folder:  /teamspace/studios/this_studio/outputs/gen_eval/atmp_AA/RANDOM_Rnd0
Agent RANDOM_Rnd1 log folder:  /teamspace/studios/this_studio/outputs/gen_eval/atmp_AA/RANDOM_Rnd1
Agent RANDOM_Rnd2 log folder:  /teamspace/studios/this_studio/outputs/gen_eval/atmp_AA/RANDOM_Rnd2
Log Name: 1780243231.510859
Log Directory: outputs/gen_eval/16-0031_DQN_DQN_vs_Rnd_eval_AA/Log/Log.log
Verbose: False
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0.
 1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.
 1. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 1.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 

[Room]: [Room][ERROR]: ---- Invalid action!


[1. 0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0.
 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 1.]
[0. 0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.
 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 

...

[0. 0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0.
 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 1.]
[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.
 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


Agent GREEDY_Grdy0 log folder:  /teamspace/studios/this_studio/outputs/gen_eval/atmp_CC/GREEDY_Grdy0
Agent GREEDY_Grdy1 log folder:  /teamspace/studios/this_studio/outputs/gen_eval/atmp_CC/GREEDY_Grdy1
Agent GREEDY_Grdy2 log folder:  /teamspace/studios/this_studio/outputs/gen_eval/atmp_CC/GREEDY_Grdy2
Log Name: 1780243440.097585
Log Directory: outputs/gen_eval/16-0400_DQN_DQN_vs_Grdy_eval_CC/Log/Log.log
Verbose: False


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


             Condition  Win Rate  Mean Pos  Mean Reward
  A: Rnd→Rnd (in-dist)      0.68      1.50        0.662
B: Rnd→Grdy (out-dist)      0.98      1.06        0.960
C: Grdy→Grdy (in-dist)      0.98      1.06        0.960


In [13]:
fig, ax = plt.subplots(figsize=(8, 5))
conditions = gen_df['Condition'].tolist()
win_rates  = gen_df['Win Rate'].tolist()
mean_pos   = gen_df['Mean Pos'].tolist()

x = np.arange(len(conditions))
w = 0.35
bars1 = ax.bar(x - w/2, win_rates, w, label='Win rate', color='steelblue')
ax2   = ax.twinx()
bars2 = ax2.bar(x + w/2, mean_pos, w, label='Mean position (lower=better)', color='darkorange', alpha=0.8)

ax.axhline(0.25, ls='--', color='grey', label='Random win rate (25%)')
ax.set_xticks(x)
ax.set_xticklabels(conditions, fontsize=9)
ax.set_ylabel('Win rate')
ax2.set_ylabel('Mean finishing position')
ax2.invert_yaxis()

lines  = [bars1, bars2, ax.get_lines()[0]]
labels = ['Win rate', 'Mean position', 'Random baseline']
ax.legend(lines, labels, loc='upper right')

ax.set_title('Experiment 2 – Generalisation Across Opponent Types', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

**8. Robustness Experiment 3 – Hyperparameter Sensitivity**

In [ ]:
HP_MATCHES = 100
decay_configs = [
    ('Fast  (0.98)',   0.98),
    ('Default (0.995)', 0.995),
    ('Slow  (0.9995)', 0.9995),
]
hp_results = []

for label, decay in decay_configs:
    set_global_seed(BASE_SEED)
    folder = f'outputs/hp_{label.split()[0].lower()}'
    mpath  = os.path.join(folder, 'model', 'dqn_model.h5')
    os.makedirs(folder, exist_ok=True)

    agent = DQNAgent(
        name=f'DQN_{label.split()[0]}',
        epsilon_decay=decay,
        training=True,
        model_path=mpath,
        log_directory=folder,
        verbose_console=False,
    )
    opps = make_random_opponents(3, os.path.join(folder, 'atmp'))
    print(f'  {label} ... ', end='', flush=True)
    res = run_game(agent, opps, HP_MATCHES, folder, room_name=f'HP_{label.split()[0]}')
    hp_results.append({'label': label, 'decay': decay, **res})
    print(f"win_rate={res['win_rate']:.3f}  mean_pos={res['mean_position']:.2f}")

print('Hyperparameter sweep complete.')


In [15]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
window = 15
palette = ['#e41a1c', '#377eb8', '#4daf4a']

for r, col in zip(hp_results, palette):
    pos  = np.array(r['positions'])
    loss = np.array(r['loss_history'])
    eps  = np.array(r['epsilon_history'])

    # Position curve
    if len(pos) >= window:
        sm = np.convolve(pos, np.ones(window)/window, mode='valid')
        axes[0].plot(sm, color=col, label=r['label'])

    # Loss curve
    axes[1].plot(loss, color=col, alpha=0.7, label=r['label'])

    # Epsilon decay trajectory
    axes[2].plot(eps, color=col, label=r['label'])

axes[0].invert_yaxis()
axes[0].set(xlabel='Match', ylabel='Mean position (lower=better)',
            title='Position by Epsilon Decay')
axes[0].legend(fontsize=8)

axes[1].set(xlabel='Replay step', ylabel='Huber loss', title='Training Loss')
axes[1].legend(fontsize=8)

axes[2].set(xlabel='Replay step', ylabel='Epsilon', title='Epsilon Trajectory')
axes[2].legend(fontsize=8)

plt.suptitle('Experiment 3 – Hyperparameter Sensitivity (Epsilon Decay)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary table
hp_df = pd.DataFrame([
    {'Config': r['label'], 'ε decay': r['decay'],
     'Win Rate': r['win_rate'], 'Mean Position': r['mean_position']}
    for r in hp_results
])
print(hp_df.to_string(index=False))

         Config  ε decay  Win Rate  Mean Position
   Fast  (0.98)   0.9800      0.49           1.87
Default (0.995)   0.9950      0.35           2.17
 Slow  (0.9995)   0.9995      0.28           2.37
